# شبکه‌های عصبی بازگشتی

در ماژول قبلی، به نمایش‌های معنایی غنی متن پرداختیم. معماری‌ای که استفاده کرده‌ایم، معنی تجمعی کلمات در یک جمله را در نظر می‌گیرد، اما ترتیب کلمات را لحاظ نمی‌کند، زیرا عملیات تجمیع که پس از توکارسازی‌ها انجام می‌شود، این اطلاعات را از متن اصلی حذف می‌کند. از آنجا که این مدل‌ها قادر به نمایش ترتیب کلمات نیستند، نمی‌توانند مسائل پیچیده‌تر یا مبهم‌تری مانند تولید متن یا پاسخ به سوال را حل کنند.

برای درک معنی یک توالی متنی، از معماری شبکه عصبی به نام **شبکه عصبی بازگشتی** یا RNN استفاده خواهیم کرد. هنگام استفاده از RNN، جمله را به صورت یک توکن در هر بار وارد شبکه می‌کنیم و شبکه حالت یا **وضعیتی** تولید می‌کند که آن را همراه با توکن بعدی مجدداً به شبکه می‌دهیم.

![تصویری که نمونه‌ای از تولید شبکه عصبی بازگشتی را نشان می‌دهد.](../../../../../translated_images/fa/rnn.27f5c29c53d727b5.webp)

با توجه به توالی ورودی توکن‌ها $X_0,\dots,X_n$، RNN توالی‌ای از بلاک‌های شبکه عصبی ایجاد می‌کند و این توالی را به صورت پایان به پایان با استفاده از پس‌انتشار آموزش می‌دهد. هر بلاک شبکه ورودی $(X_i,S_i)$ را دریافت می‌کند و خروجی $S_{i+1}$ را تولید می‌کند. حالت نهایی $S_n$ یا خروجی $Y_n$ وارد یک طبقه‌بند خطی شده تا نتیجه نهایی تولید شود. همه بلاک‌های شبکه وزن‌های مشترکی دارند و با یک بار عبور پس‌انتشار به صورت پایان به پایان آموزش داده می‌شوند.

> شکل بالا شبکه عصبی بازگشتی را در حالت باز شده (سمت چپ) و حالت فشرده‌تر بازگشتی (سمت راست) نشان می‌دهد. مهم است بدانید که تمام سلول‌های RNN دارای وزن‌های **قابل اشتراک‌گذاری یکسان** هستند.

از آنجا که بردارهای حالت $S_0,\dots,S_n$ در سراسر شبکه منتقل می‌شوند، RNN قادر است وابستگی‌های ترتیبی بین کلمات را یاد بگیرد. برای مثال، هنگامی که کلمه *not* در جایی از توالی ظاهر شود، می‌تواند یاد بگیرد که برخی عناصر در بردار حالت را نفی کند.

در داخل، هر سلول RNN شامل دو ماتریس وزن $W_H$ و $W_I$ و بایاس $b$ است. در هر گام RNN، با داشتن ورودی $X_i$ و حالت ورودی $S_i$، حالت خروجی به صورت $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$ محاسبه می‌شود که $f$ تابع فعال‌سازی است (معمولاً $\tanh$).

> برای مسائلی مانند تولید متن (که در واحد بعدی بررسی خواهیم کرد) یا ترجمه ماشینی، ما همچنین می‌خواهیم در هر گام RNN مقداری خروجی دریافت کنیم. در این حالت، ماتریس دیگری به نام $W_O$ نیز وجود دارد و خروجی به صورت $Y_i=f(W_O\times S_i+b_O)$ محاسبه می‌شود.

بیایید ببینیم چگونه شبکه‌های عصبی بازگشتی می‌توانند به ما در طبقه‌بندی مجموعه داده خبری کمک کنند.

> برای محیط سندباکس، باید سلول زیر را اجرا کنیم تا مطمئن شویم کتابخانه مورد نیاز نصب شده و داده‌ها پیش‌بارگذاری شده‌اند. اگر به صورت محلی اجرا می‌کنید، می‌توانید این سلول را رد کنید.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

هنگام آموزش مدل‌های بزرگ، تخصیص حافظه GPU ممکن است به مشکل برخورد کند. همچنین ممکن است بخواهیم با اندازه‌های مختلف مینی‌باتچ آزمایش کنیم، به طوری که داده‌ها در حافظه GPU ما جا شود، اما آموزش به اندازه کافی سریع باشد. اگر این کد را روی دستگاه GPU خود اجرا می‌کنید، می‌توانید با تنظیم اندازه مینی‌باتچ آزمایش کنید تا آموزش سریع‌تر شود.

> **توجه**: برخی نسخه‌های درایورهای NVidia شناخته شده‌اند که پس از آموزش مدل حافظه را آزاد نمی‌کنند. ما چندین مثال را در این دفترچه اجرا می‌کنیم و این ممکن است باعث پر شدن حافظه در برخی تنظیمات شود، به ویژه اگر به عنوان بخشی از همان دفترچه آزمایش‌های خود را انجام می‌دهید. اگر هنگام شروع آموزش مدل با خطاهای عجیبی مواجه شدید، ممکن است بخواهید کرنل دفترچه را ری‌استارت کنید.


In [3]:
batch_size = 16
embed_size = 64

## طبقه‌بند ساده RNN

در مورد یک RNN ساده، هر واحد بازگشتی یک شبکه خطی ساده است که یک بردار ورودی و یک بردار حالت می‌گیرد و یک بردار حالت جدید تولید می‌کند. در کتابخانه Keras، این را می‌توان با لایه `SimpleRNN` نشان داد.

در حالی که می‌توانیم توکن‌های انکد شده به صورت One-hot را مستقیماً به لایه RNN ارسال کنیم، این ایده خوبی نیست به دلیل بعد بالا آنها. بنابراین، از یک لایه embedding برای کاهش بعد بردارهای کلمات استفاده می‌کنیم، سپس یک لایه RNN داریم و در نهایت یک طبقه‌بند `Dense`.

> **نکته**: در مواردی که بعد داده خیلی بالا نیست، برای مثال هنگام استفاده از توکن‌سازی سطح کاراکتر، ممکن است منطقی باشد که توکن‌های انکد شده به صورت One-hot را مستقیماً به سلول RNN بفرستیم.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **توجه:** ما در اینجا برای سادگی از یک لایه جاسازی آموزش‌دیده نشده استفاده می‌کنیم، اما برای نتایج بهتر می‌توانیم از یک لایه جاسازی آموزش‌دیده‌شده با استفاده از Word2Vec استفاده کنیم، همان‌طور که در واحد قبلی توضیح داده شد. این می‌تواند یک تمرین خوب برای شما باشد که این کد را برای کار با جاسازی‌های آموزش‌دیده‌شده تطبیق دهید.

حال بیایید RNN خود را آموزش دهیم. آموزش RNNها به طور کلی دشوار است، زیرا وقتی سلول‌های RNN در طول دنباله باز می‌شوند، تعداد لایه‌های درگیر در پس‌انتشار خطا بسیار زیاد می‌شود. بنابراین باید نرخ یادگیری کوچک‌تری انتخاب کنیم و شبکه را روی مجموعه داده بزرگ‌تر آموزش دهیم تا نتایج خوبی به دست آید. این ممکن است زمان زیادی ببرد، بنابراین استفاده از GPU ترجیح داده می‌شود.

برای سرعت بخشیدن، مدل RNN را فقط روی عناوین اخبار آموزش می‌دهیم و توضیحات را حذف می‌کنیم. شما می‌توانید با آموزش همراه با توضیحات امتحان کنید و ببینید آیا می‌توانید مدل را آموزش دهید یا خیر.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **توجه** داشته باشید که دقت احتمالاً در اینجا کمتر است، زیرا ما فقط بر روی عناوین اخبار آموزش می‌دهیم.


## بازنگری توالی‌های متغیر

به یاد داشته باشید که لایه `TextVectorization` به طور خودکار توالی‌های با طول متغیر در یک مینی‌بتچ را با توکن‌های پرکننده عرض می‌کند. مشخص شده است که این توکن‌ها نیز در آموزش شرکت می‌کنند، و می‌توانند همگرایی مدل را پیچیده کنند.

چندین رویکرد وجود دارد که می‌توانیم برای حداقل کردن میزان پرکننده‌ها به کار ببریم. یکی از آنها مرتب‌سازی مجدد داده‌ها بر اساس طول توالی است و گروه‌بندی تمام توالی‌ها بر اساس اندازه. این کار را می‌توان با استفاده از تابع `tf.data.experimental.bucket_by_sequence_length` انجام داد (نگاه کنید به [مستندات](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)).

رویکرد دیگری استفاده از **ماسک‌گذاری** است. در کرَس، برخی از لایه‌ها ورودی اضافی را پشتیبانی می‌کنند که نشان می‌دهد کدام توکن‌ها باید هنگام آموزش در نظر گرفته شوند. برای وارد کردن ماسک‌گذاری به مدل خود، می‌توانیم یا یک لایه جداگانه `Masking` اضافه کنیم ([مستندات](https://keras.io/api/layers/core_layers/masking/))، یا می‌توانیم پارامتر `mask_zero=True` لایه `Embedding` خود را مشخص کنیم.

> **توجه**: این آموزش حدود ۵ دقیقه طول می‌کشد تا یک اپوک کامل روی کل داده‌ها انجام شود. اگر صبرتان تمام شد، می‌توانید در هر زمانی آموزش را متوقف کنید. کاری که می‌توانید انجام دهید این است که مقدار داده‌های استفاده شده برای آموزش را محدود کنید، با اضافه کردن بند `.take(...)` بعد از داده‌های `ds_train` و `ds_test`.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


اکنون که از ماسک‌گذاری استفاده می‌کنیم، می‌توانیم مدل را روی کل مجموعه داده عناوین و توضیحات آموزش دهیم.

> **توجه**: آیا متوجه شده‌اید که ما از بردارساز آموزش‌دیده روی عناوین خبری استفاده کرده‌ایم و نه کل متن مقاله؟ این می‌تواند باعث شود برخی از توکن‌ها نادیده گرفته شوند، بنابراین بهتر است بردارساز مجدداً آموزش داده شود. با این حال، ممکن است تأثیر آن بسیار اندک باشد، بنابراین برای سادگی به بردارساز پیش‌آموزش‌دیده قبلی پایبند می‌مانیم.


## LSTM: حافظه بلندمدت کوتاه‌مدت

یکی از مشکلات اصلی شبکه‌های عصبی بازگشتی (RNN) مسئله **ناپدید شدن گرادیان‌ها** است. شبکه‌های RNN ممکن است خیلی طولانی باشند و در طول فرآیند پس‌انتشار نتوانند گرادیان‌ها را تا لایه اول شبکه به خوبی منتقل کنند. هنگامی که این اتفاق می‌افتد، شبکه نمی‌تواند روابط بین نشانه‌های دورافتاده را یاد بگیرد. یک راه برای جلوگیری از این مشکل، معرفی **مدیریت حالت صریح** با استفاده از **دروازه‌ها** است. دو معماری رایج که دروازه‌ها را معرفی می‌کنند، **حافظه بلندمدت کوتاه‌مدت** (LSTM) و **واحد رله دروازه‌ای** (GRU) هستند. ما در اینجا LSTMها را پوشش می‌دهیم.

![تصویری که نمونه‌ای از سلول حافظه بلندمدت کوتاه‌مدت را نشان می‌دهد](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

شبکه LSTM به گونه‌ای مشابه RNN سازماندهی شده است، اما دو حالت وجود دارد که از لایه‌ای به لایه دیگر منتقل می‌شود: حالت واقعی $c$ و بردار مخفی $h$. در هر واحد، بردار مخفی $h_{t-1}$ با ورودی $x_t$ ترکیب می‌شود و با هم کنترل می‌کنند چه اتفاقی برای حالت $c_t$ و خروجی $h_t$ از طریق **دروازه‌ها** می‌افتد. هر دروازه دارای تابع فعال‌سازی سیگموید است (خروجی در بازه $[0,1]$) که می‌توان آن را به عنوان یک ماسک بیت به بیت هنگام ضرب در بردار حالت تصور کرد. LSTMها دارای دروازه‌های زیر هستند (از چپ به راست در تصویر بالا):
* **دروازه فراموشی** که تعیین می‌کند کدام اجزای بردار $c_{t-1}$ باید فراموش شوند و کدام باید عبور داده شوند.
* **دروازه ورودی** که تعیین می‌کند چه مقدار اطلاعات از بردار ورودی و بردار مخفی قبلی باید به بردار حالت وارد شود.
* **دروازه خروجی** که بردار حالت جدید را گرفته و تصمیم می‌گیرد کدام از اجزای آن برای تولید بردار مخفی جدید $h_t$ استفاده شود.

اجزای حالت $c$ را می‌توان به عنوان پرچم‌هایی تصور کرد که می‌توانند روشن و خاموش شوند. برای مثال، وقتی نام *آلیس* را در توالی می‌بینیم، حدس می‌زنیم که به یک زن اشاره دارد و پرچمی را در حالت بالا می‌بریم که می‌گوید جمله شامل یک اسم مونث است. وقتی بعداً کلمات *و تام* را می‌بینیم، پرچمی را می‌بریم که می‌گوید جمله شامل اسم جمع است. به این ترتیب با دستکاری حالت می‌توانیم ویژگی‌های دستوری جمله را دنبال کنیم.

> **توجه**: این منبع بسیار خوبی برای درک جزئیات داخلی LSTM است: [درک شبکه‌های LSTM](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) توسط کریستوفر اولاه.

در حالی که ساختار داخلی یک سلول LSTM ممکن است پیچیده به نظر برسد، کتابخانه Keras این پیاده‌سازی را داخل لایه `LSTM` مخفی کرده است، بنابراین تنها کاری که باید در مثال بالا انجام دهیم، جایگزینی لایه بازگشتی است:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **توجه** داشته باشید که آموزش LSTMها نیز نسبتاً کند است و ممکن است در ابتدای آموزش افزایش قابل‌توجهی در دقت مشاهده نکنید. ممکن است نیاز باشد برای مدتی آموزش را ادامه دهید تا به دقت خوبی برسید.


## شبکه‌های بازگشتی دوطرفه و چندلایه

در مثال‌های قبلی، شبکه‌های بازگشتی از ابتدای یک دنباله تا انتهای آن کار می‌کنند. این برای ما طبیعی به نظر می‌رسد چون هم‌جهت با جهتی است که ما متن را می‌خوانیم یا به صحبت گوش می‌دهیم. با این حال، برای سناریوهایی که نیاز به دسترسی تصادفی به ورودی دنباله دارند، منطقی‌تر است که محاسبه بازگشتی در هر دو جهت انجام شود. شبکه‌های بازگشتی که اجازه محاسبه در هر دو جهت را می‌دهند، شبکه‌های بازگشتی **دوطرفه** نامیده می‌شوند و می‌توان آن‌ها را با بسته‌بندی لایه بازگشتی با یک لایه ویژه `Bidirectonal` ساخت.

> **توجه**: لایه `Bidirectional` دو نسخه از لایه داخل خود می‌سازد، و ویژگی `go_backwards` یکی از این نسخه‌ها را روی `True` تنظیم می‌کند، که باعث می‌شود در جهت مخالف دنباله حرکت کند.

شبکه‌های بازگشتی، چه یک‌جهته و چه دوطرفه، الگوهای داخل یک دنباله را استخراج کرده و آن‌ها را در وکتورهای حالت ذخیره می‌کنند یا به عنوان خروجی باز می‌گردانند. مانند شبکه‌های کانولوشنی، می‌توانیم لایه بازگشتی دیگری پس از اولین لایه اضافه کنیم تا الگوهای سطح بالاتر که از الگوهای سطح پایین استخراج‌شده توسط لایه اول ساخته شده‌اند را استخراج کنیم. این ما را به مفهوم **شبکه بازگشتی چندلایه** می‌رساند که شامل دو یا چند شبکه بازگشتی است که خروجی لایه قبلی به عنوان ورودی به لایه بعدی داده می‌شود.

![تصویری از شبکه RNN با حافظه بلندمدت چندلایه](../../../../../translated_images/fa/multi-layer-lstm.dd975e29bb2a59fe.webp)

*تصویر برگرفته از [این نوشته عالی](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) توسط فرناندو لوپز.*

کار ساخت این شبکه‌ها در کرِراس بسیار آسان است، چون فقط کافی است لایه‌های بازگشتی بیشتری به مدل اضافه کنید. برای همه لایه‌ها به جز آخرین لایه، باید پارامتر `return_sequences=True` را تعیین کنیم، چون نیاز داریم لایه همه حالت‌های میانی را بازگرداند، نه فقط حالت نهایی محاسبه بازگشتی.

بیایید یک LSTM دو لایه دوطرفه برای مسئله دسته‌بندی خود بسازیم.

> **توجه** این کد نیز مدت زمان زیادی طول می‌کشد تا اجرا شود، اما بالاترین دقتی را که تاکنون دیده‌ایم به ما می‌دهد. پس شاید ارزشش را داشته باشد که منتظر بمانیم و نتیجه را ببینیم.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## شبکه‌های بازگشتی برای وظایف دیگر

تا اینجا، تمرکز ما بر استفاده از شبکه‌های بازگشتی برای دسته‌بندی دنباله‌های متنی بوده است. اما آن‌ها می‌توانند وظایف بسیار بیشتری مانند تولید متن و ترجمه ماشینی را انجام دهند — در واحد بعدی به این وظایف خواهیم پرداخت.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
